In [ ]:
# ============================================================
# EXPLORACIÓN INICIAL DE BASES DENUE
# Proyecto: Nearshoring_Project
# Objetivo: visualizar muestras y columnas disponibles
# ============================================================

# ------------------------------------------------------------
# 0. Montar Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 1. Librerías
# ------------------------------------------------------------

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 2. Definir ruta base
# ------------------------------------------------------------

# Si tu carpeta se llama DENUE en lugar de DNUE, cambia "DNUE" por "DENUE"
base_path = Path("/content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE")


# ------------------------------------------------------------
# 3. Definir archivos DENUE
# ------------------------------------------------------------

denue_paths = {
    # 2023
    "denue_31_33_2023": base_path / "2023" / "denue_31_33_2023.csv",
    "denue_48_49_2023": base_path / "2023" / "denue_48_49_2023.csv",
    "denue_54_2023":    base_path / "2023" / "denue_54_2023.csv",
    "denue_56_2023":    base_path / "2023" / "denue_56_2023.csv",

    # 2018
    "denue_31_33_2018": base_path / "2018" / "denue_31_33_2018.csv",
    "denue_48_49_2018": base_path / "2018" / "denue_48_49_2018.csv",
    "denue_54_2018":    base_path / "2018" / "denue_54_2018.csv",
    "denue_56_2018":    base_path / "2018" / "denue_56_2018.csv",
}


# ------------------------------------------------------------
# 4. Verificar que los archivos existan
# ------------------------------------------------------------

print("VERIFICACIÓN DE ARCHIVOS")
print("=" * 100)

for name, path in denue_paths.items():
    status = "OK" if path.exists() else "NO ENCONTRADO"
    print(f"{name}: {status}")
    print(f"Ruta: {path}")
    print("-" * 100)


# ------------------------------------------------------------
# 5. Función para leer una muestra pequeña de cada archivo
# ------------------------------------------------------------

def preview_denue_file(file_path, n_rows=5):
    """
    Lee una pequeña muestra de un archivo DENUE.
    No carga toda la base completa, solo las primeras filas.
    Esto sirve para explorar columnas sin consumir demasiada memoria.
    """

    try:
        df_sample = pd.read_csv(
            file_path,
            nrows=n_rows,
            dtype=str,
            encoding="latin1"
        )

        print(f"\nArchivo: {file_path.name}")
        print(f"Número de filas leídas: {df_sample.shape[0]}")
        print(f"Número de columnas: {df_sample.shape[1]}")

        print("\nColumnas disponibles:")
        for i, col in enumerate(df_sample.columns, start=1):
            print(f"{i}. {col}")

        print("\nMuestra de datos:")
        display(df_sample)

        print("=" * 100)

        return df_sample

    except UnicodeDecodeError:
        print(f"\nError de codificación con latin1 en: {file_path.name}")
        print("Intentando con utf-8...")

        try:
            df_sample = pd.read_csv(
                file_path,
                nrows=n_rows,
                dtype=str,
                encoding="utf-8"
            )

            print(f"\nArchivo: {file_path.name}")
            print(f"Número de filas leídas: {df_sample.shape[0]}")
            print(f"Número de columnas: {df_sample.shape[1]}")

            print("\nColumnas disponibles:")
            for i, col in enumerate(df_sample.columns, start=1):
                print(f"{i}. {col}")

            print("\nMuestra de datos:")
            display(df_sample)

            print("=" * 100)

            return df_sample

        except Exception as e:
            print(f"No se pudo leer el archivo: {file_path.name}")
            print(e)
            print("=" * 100)
            return None

    except Exception as e:
        print(f"\nError leyendo archivo: {file_path.name}")
        print(e)
        print("=" * 100)
        return None


# ------------------------------------------------------------
# 6. Visualizar una muestra de cada base
# ------------------------------------------------------------

samples = {}

print("\n\nEXPLORACIÓN DE MUESTRAS")
print("=" * 100)

for name, path in denue_paths.items():

    print(f"\nExplorando: {name}")
    print("-" * 100)

    if path.exists():
        samples[name] = preview_denue_file(path, n_rows=5)
    else:
        print(f"No se encontró el archivo:")
        print(path)
        samples[name] = None


# ------------------------------------------------------------
# 7. Crear resumen largo de columnas por archivo
# ------------------------------------------------------------

columns_summary = []

for file_name, df in samples.items():
    if df is not None:
        for col in df.columns:
            columns_summary.append({
                "file": file_name,
                "column": col
            })

columns_summary_df = pd.DataFrame(columns_summary)

print("\n\nRESUMEN LARGO DE COLUMNAS POR ARCHIVO")
print("=" * 100)

display(columns_summary_df)


# ------------------------------------------------------------
# 8. Crear matriz de presencia de columnas
# ------------------------------------------------------------

if not columns_summary_df.empty:

    column_presence = (
        columns_summary_df
        .assign(present=1)
        .pivot_table(
            index="column",
            columns="file",
            values="present",
            fill_value=0
        )
        .reset_index()
    )

    print("\n\nMATRIZ DE PRESENCIA DE COLUMNAS")
    print("=" * 100)

    display(column_presence)

else:
    print("No se pudo crear la matriz de columnas porque no se leyó ningún archivo.")


# ------------------------------------------------------------
# 9. Resumen rápido de columnas por base
# ------------------------------------------------------------

print("\n\nLISTADO DE COLUMNAS POR BASE")
print("=" * 100)

for file_name, df in samples.items():
    print(f"\n{file_name}")

    if df is not None:
        print(f"Total de columnas: {len(df.columns)}")
        print(list(df.columns))
    else:
        print("No disponible")


# ------------------------------------------------------------
# 10. Identificar columnas comunes a todas las bases
# ------------------------------------------------------------

valid_samples = {name: df for name, df in samples.items() if df is not None}

if len(valid_samples) > 0:

    column_sets = [set(df.columns) for df in valid_samples.values()]

    common_columns = set.intersection(*column_sets)
    all_columns = set.union(*column_sets)

    print("\n\nCOLUMNAS COMUNES A TODAS LAS BASES")
    print("=" * 100)
    print(f"Número de columnas comunes: {len(common_columns)}")
    print(sorted(common_columns))

    print("\n\nTODAS LAS COLUMNAS ENCONTRADAS")
    print("=" * 100)
    print(f"Número total de columnas distintas: {len(all_columns)}")
    print(sorted(all_columns))

else:
    print("No hay bases válidas para comparar columnas.")


# ------------------------------------------------------------
# 11. Revisar tamaño aproximado de archivos en MB
# ------------------------------------------------------------

file_sizes = []

for name, path in denue_paths.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        file_sizes.append({
            "file": name,
            "size_mb": round(size_mb, 2),
            "path": str(path)
        })
    else:
        file_sizes.append({
            "file": name,
            "size_mb": None,
            "path": str(path)
        })

file_sizes_df = pd.DataFrame(file_sizes)

print("\n\nTAMAÑO DE ARCHIVOS")
print("=" * 100)

display(file_sizes_df)


# ------------------------------------------------------------
# 12. Guardar resumen de exploración en outputs opcionalmente
# ------------------------------------------------------------

output_folder = Path("/content/drive/MyDrive/Nearshoring_Project/outputs/tables")
output_folder.mkdir(parents=True, exist_ok=True)

columns_summary_path = output_folder / "denue_columns_summary.csv"
column_presence_path = output_folder / "denue_column_presence_matrix.csv"
file_sizes_path = output_folder / "denue_raw_file_sizes.csv"

columns_summary_df.to_csv(columns_summary_path, index=False, encoding="utf-8-sig")

if "column_presence" in locals():
    column_presence.to_csv(column_presence_path, index=False, encoding="utf-8-sig")

file_sizes_df.to_csv(file_sizes_path, index=False, encoding="utf-8-sig")

print("\n\nARCHIVOS DE RESUMEN GUARDADOS")
print("=" * 100)
print(columns_summary_path)
print(column_presence_path)
print(file_sizes_path)

Mounted at /content/drive
VERIFICACIÓN DE ARCHIVOS
denue_31_33_2023: OK
Ruta: /content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE/2023/denue_31_33_2023.csv
----------------------------------------------------------------------------------------------------
denue_48_49_2023: OK
Ruta: /content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE/2023/denue_48_49_2023.csv
----------------------------------------------------------------------------------------------------
denue_54_2023: OK
Ruta: /content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE/2023/denue_54_2023.csv
----------------------------------------------------------------------------------------------------
denue_56_2023: OK
Ruta: /content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE/2023/denue_56_2023.csv
----------------------------------------------------------------------------------------------------
denue_31_33_2018: OK
Ruta: /content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE/2018/denue_31_33_2018.csv
-------

,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,7239103,01003311993000011000000000U0,PREPARACION Y VENTA DE SALSAS Y FRIJOLES,NaN,311993,ElaboraciÃ³n de alimentos frescos para consumo...,0 a 5 personas,CALLE,20 DE NOVIEMBRE,CALLE,...,Calvillo,0219,026,NaN,NaN,NaN,Fijo,21.84339063999999908,-102.72191499999999564,2019-11
1,1079,01003312112000042000000000U1,PROCESADORA DE AGUA BIOS,AGUA PURIFICADA BIOS,312112,PurificaciÃ³n y embotellado de agua,0 a 5 personas,AVENIDA,ENRIQUE OLIVARES SANTANA,CALLE,...,Ojocaliente,0079,003,NaN,NaN,NaN,Fijo,21.87640830000000136,-102.67415630000000704,2010-07
2,7462448,01005311993000031000000000U3,POSTRES JESUS MARIA,NaN,311993,ElaboraciÃ³n de alimentos frescos para consumo...,0 a 5 personas,CALLE,PIRULES,CALLE,...,JesÃºs MarÃ­a,0548,013,NaN,NaN,NaN,Fijo,21.95989047999999855,-102.34880228999999474,2019-11
3,13330,01001311613000041000000000U5,PREPARACION DE EMBUTIDOS Y OTRAS CONSERVAS,NaN,311613,PreparaciÃ³n de embutidos y otras conservas de...,0 a 5 personas,CALLE,FELIPE ANGELES,CALLE,...,Aguascalientes,2460,026,NaN,VILLA-H@LIVE.COM.MX,NaN,Fijo,21.86814062999999919,-102.27597828000000391,2010-07
4,8387930,01005311910000124000000000U0,PROCESADORA DE BOTANAS Y ENCURTIDOS DE AGUASCA...,PROCESADORA DE BOTANAS Y ENCURTIDOS DE AGUASCA...,311910,ElaboraciÃ³n de botanas,31 a 50 personas,AVENIDA,MUEBLEROS,AVENIDA,...,Parque Industrial Chichimeco (PICH),0016,008,NaN,PROCESADORA2009@HOTMAIL.COM,NaN,Fijo,21.98945005999999935,-102.34188982000000578,2019-11



Explorando: denue_48_49_2023
----------------------------------------------------------------------------------------------------

Archivo: denue_48_49_2023.csv
Número de filas leídas: 5
Número de columnas: 42

Columnas disponibles:
1. id
2. clee
3. nom_estab
4. raz_social
5. codigo_act
6. nombre_act
7. per_ocu
8. tipo_vial
9. nom_vial
10. tipo_v_e_1
11. nom_v_e_1
12. tipo_v_e_2
13. nom_v_e_2
14. tipo_v_e_3
15. nom_v_e_3
16. numero_ext
17. letra_ext
18. edificio
19. edificio_e
20. numero_int
21. letra_int
22. tipo_asent
23. nomb_asent
24. tipoCenCom
25. nom_CenCom
26. num_local
27. cod_postal
28. cve_ent
29. entidad
30. cve_mun
31. municipio
32. cve_loc
33. localidad
34. ageb
35. manzana
36. telefono
37. correoelec
38. www
39. tipoUniEco
40. latitud
41. longitud
42. fecha_alta

Muestra de datos:


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,9317111,01001484210000171000000000U4,ABA MUDANZA,NaN,484210,Servicios de mudanzas,6 a 10 personas,CALLE,Ébano,CALLE,...,Aguascalientes,1509,045,NaN,CONTACTO@ABAMUDANZAS.COM.MX,WWW.ABAMUDANZAS.COM.MX,Fijo,21.90368602,-102.29866397,2020-11
1,50830,01001481111000032000004510S3,AEROMEXICO,AEROVIAS DE MEXICO SA DE CV,481111,Transporte aéreo regular en líneas aéreas naci...,0 a 5 personas,AVENIDA,INDEPENDENCIA,CALLE,...,Aguascalientes,0229,002,4494484019,FOBAGUGALERIAS@AEROMEXICO.COM,NaN,Fijo,21.92330064,-102.29779263,2014-12
2,9302601,01001481111000072000004510S9,AEROMEXICO,AEROVIAS DE MEXICO SA DE CV,481111,Transporte aéreo regular en líneas aéreas naci...,6 a 10 personas,AVENIDA,HÉROE DE NACOZARI SUR,CALLE,...,Aguascalientes,159A,017,NaN,NaN,NaN,Fijo,21.85704963,-102.28282318,2020-11
3,9435786,01001481111000082000000000U8,AEROMEXICO,ESPECIALISTAS AEREOS DE AGUASCALIENTES SA DE CV,481111,Transporte aéreo regular en líneas aéreas naci...,6 a 10 personas,AVENIDA,INDEPENDENCIA,BOULEVARD,...,Aguascalientes,0229,002,4494484019,FOBAGUGALERIAS@AEROMEXICO.COM,WWW.AEROMEXICO.COM,Fijo,21.92366051,-102.2978316,2023-11
4,34921,01001481111000041000013985S4,AEROMEXICO CARGO,NaN,481111,Transporte aéreo regular en líneas aéreas naci...,0 a 5 personas,OTRO(ESPECIFIQUE),NINGUNO,OTRO(ESPECIFIQUE),...,Aguascalientes,0498,023,NaN,NaN,NaN,Fijo,21.89144571,-102.30195433,2014-12



Explorando: denue_54_2023
----------------------------------------------------------------------------------------------------

Archivo: denue_54_2023.csv
Número de filas leídas: 5
Número de columnas: 42

Columnas disponibles:
1. id
2. clee
3. nom_estab
4. raz_social
5. codigo_act
6. nombre_act
7. per_ocu
8. tipo_vial
9. nom_vial
10. tipo_v_e_1
11. nom_v_e_1
12. tipo_v_e_2
13. nom_v_e_2
14. tipo_v_e_3
15. nom_v_e_3
16. numero_ext
17. letra_ext
18. edificio
19. edificio_e
20. numero_int
21. letra_int
22. tipo_asent
23. nomb_asent
24. tipoCenCom
25. nom_CenCom
26. num_local
27. cod_postal
28. cve_ent
29. entidad
30. cve_mun
31. municipio
32. cve_loc
33. localidad
34. ageb
35. manzana
36. telefono
37. correoelec
38. www
39. tipoUniEco
40. latitud
41. longitud
42. fecha_alta

Muestra de datos:


...



MATRIZ DE PRESENCIA DE COLUMNAS


file,column,denue_31_33_2018,denue_31_33_2023,denue_48_49_2018,denue_48_49_2023,denue_54_2018,denue_54_2023,denue_56_2018,denue_56_2023
0,ageb,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,clee,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
2,cod_postal,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,codigo_act,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,correoelec,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,cve_ent,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,cve_loc,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,cve_mun,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,edificio,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,edificio_e,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0




LISTADO DE COLUMNAS POR BASE

denue_31_33_2023
Total de columnas: 42
['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'entidad', 'cve_mun', 'municipio', 'cve_loc', 'localidad', 'ageb', 'manzana', 'telefono', 'correoelec', 'www', 'tipoUniEco', 'latitud', 'longitud', 'fecha_alta']

denue_48_49_2023
Total de columnas: 42
['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'enti

,file,size_mb,path
0,denue_31_33_2023,238.43,/content/drive/MyDrive/Nearshoring_Project/dat...
1,denue_48_49_2023,21.91,/content/drive/MyDrive/Nearshoring_Project/dat...
2,denue_54_2023,55.94,/content/drive/MyDrive/Nearshoring_Project/dat...
3,denue_56_2023,31.34,/content/drive/MyDrive/Nearshoring_Project/dat...
4,denue_31_33_2018,202.01,/content/drive/MyDrive/Nearshoring_Project/dat...
5,denue_48_49_2018,16.11,/content/drive/MyDrive/Nearshoring_Project/dat...
6,denue_54_2018,40.78,/content/drive/MyDrive/Nearshoring_Project/dat...
7,denue_56_2018,50.31,/content/drive/MyDrive/Nearshoring_Project/dat...




ARCHIVOS DE RESUMEN GUARDADOS
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_columns_summary.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_column_presence_matrix.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_raw_file_sizes.csv


In [ ]:
# ============================================================
# CREACIÓN DE BASE ÚNICA DENUE
# Proyecto: Nearshoring_Project
#
# Objetivo:
# Unir y limpiar las bases DENUE 2018 y 2023 para construir
# una sola base de trabajo a nivel establecimiento.
#
# Sectores incluidos:
# - 31-33 Industrias manufactureras
# - 48-49 Transportes, correos y almacenamiento
# - 54 Servicios profesionales, científicos y técnicos
# - 56 Servicios de apoyo a los negocios y manejo de desechos
#
# Output principal:
# data/processed/denue_industrial_support_base.csv
#
# Unidad de observación:
# year × establishment
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 1. Librerías
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path
import re


# ------------------------------------------------------------
# 2. Definir rutas del proyecto
# ------------------------------------------------------------

# Ojo: en tus carpetas aparece DNUE. Si después corriges a DENUE,
# cambia esta ruta.
raw_denue_path = Path("/content/drive/MyDrive/Nearshoring_Project/data/raw/DNUE")

processed_path = Path("/content/drive/MyDrive/Nearshoring_Project/data/processed")
tables_path = Path("/content/drive/MyDrive/Nearshoring_Project/outputs/tables")

processed_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Definir archivos DENUE a integrar
# ------------------------------------------------------------

denue_files = [
    # 2023
    raw_denue_path / "2023" / "denue_31_33_2023.csv",
    raw_denue_path / "2023" / "denue_48_49_2023.csv",
    raw_denue_path / "2023" / "denue_54_2023.csv",
    raw_denue_path / "2023" / "denue_56_2023.csv",

    # 2018
    raw_denue_path / "2018" / "denue_31_33_2018.csv",
    raw_denue_path / "2018" / "denue_48_49_2018.csv",
    raw_denue_path / "2018" / "denue_54_2018.csv",
    raw_denue_path / "2018" / "denue_56_2018.csv",
]


# ------------------------------------------------------------
# 4. Diccionarios de sectores descargados
# ------------------------------------------------------------

sector_labels = {
    "31_33": "Industrias manufactureras",
    "48_49": "Transportes, correos y almacenamiento",
    "54": "Servicios profesionales, científicos y técnicos",
    "56": "Servicios de apoyo a los negocios y manejo de desechos y servicios de remediación"
}

sector_short_labels = {
    "31_33": "manufacturing",
    "48_49": "transport_logistics_storage",
    "54": "professional_scientific_technical_services",
    "56": "business_support_waste_remediation_services"
}


# ------------------------------------------------------------
# 5. Funciones auxiliares
# ------------------------------------------------------------

def read_denue_csv(file_path):
    """
    Lee un archivo CSV de DENUE.
    Mantiene todo como texto para no perder ceros a la izquierda
    en claves geográficas o códigos.
    """

    encodings_to_try = ["utf-8", "utf-8-sig", "latin1"]

    for enc in encodings_to_try:
        try:
            df = pd.read_csv(
                file_path,
                dtype=str,
                encoding=enc,
                low_memory=False
            )
            print(f"Archivo leído con encoding {enc}: {file_path.name}")
            return df

        except UnicodeDecodeError:
            continue

    raise ValueError(f"No se pudo leer el archivo: {file_path}")


def extract_year_from_filename(file_name):
    """
    Extrae el año desde nombres como:
    denue_31_33_2023.csv
    denue_54_2018.csv
    """

    match = re.search(r"(2018|2023)", file_name)

    if match:
        return int(match.group(1))

    return np.nan


def extract_sector_from_filename(file_name):
    """
    Extrae el bloque sectorial desde el nombre del archivo.
    """

    if "31_33" in file_name:
        return "31_33"
    elif "48_49" in file_name:
        return "48_49"
    elif "54" in file_name:
        return "54"
    elif "56" in file_name:
        return "56"
    else:
        return "unknown"


def clean_text_column(series):
    """
    Limpieza básica de columnas de texto.
    """

    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ------------------------------------------------------------
# 6. Leer, etiquetar y unir archivos
# ------------------------------------------------------------

denue_list = []

print("LECTURA DE ARCHIVOS DENUE")
print("=" * 100)

for file_path in denue_files:

    if not file_path.exists():
        print(f"NO ENCONTRADO: {file_path}")
        continue

    file_name = file_path.name

    year = extract_year_from_filename(file_name)
    sector_code = extract_sector_from_filename(file_name)

    sector_label = sector_labels.get(sector_code, "Unknown")
    sector_short_label = sector_short_labels.get(sector_code, "unknown")

    df = read_denue_csv(file_path)

    # Homologar estructura:
    # Los archivos 2023 tienen clee, los de 2018 no.
    if "clee" not in df.columns:
        df["clee"] = pd.NA

    # Agregar metadatos desde el nombre del archivo
    df["year"] = year
    df["denue_sector_block"] = sector_code
    df["denue_sector_label"] = sector_label
    df["denue_sector_short_label"] = sector_short_label
    df["source_file"] = file_name

    denue_list.append(df)

    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    print("-" * 100)


denue_raw_combined = pd.concat(denue_list, ignore_index=True)

print("\nBASE UNIDA SIN LIMPIEZA FINAL")
print("=" * 100)
print(f"Filas totales: {denue_raw_combined.shape[0]:,}")
print(f"Columnas totales: {denue_raw_combined.shape[1]:,}")


# ------------------------------------------------------------
# 7. Seleccionar columnas útiles
# ------------------------------------------------------------

columns_to_keep = [
    # Metadatos del proyecto
    "year",
    "denue_sector_block",
    "denue_sector_label",
    "denue_sector_short_label",
    "source_file",

    # Identificación del establecimiento
    "id",
    "clee",
    "nom_estab",
    "raz_social",
    "tipoUniEco",

    # Actividad económica
    "codigo_act",
    "nombre_act",
    "per_ocu",

    # Geografía
    "cve_ent",
    "entidad",
    "cve_mun",
    "municipio",
    "cve_loc",
    "localidad",
    "ageb",
    "manzana",
    "latitud",
    "longitud",

    # Dirección básica
    "tipo_vial",
    "nom_vial",
    "numero_ext",
    "letra_ext",
    "numero_int",
    "letra_int",
    "tipo_asent",
    "nomb_asent",
    "cod_postal",

    # Contacto
    "telefono",
    "correoelec",
    "www",

    # Fecha de alta
    "fecha_alta"
]

columns_existing = [col for col in columns_to_keep if col in denue_raw_combined.columns]

denue_clean = denue_raw_combined[columns_existing].copy()


# ------------------------------------------------------------
# 8. Limpieza de claves geográficas
# ------------------------------------------------------------

# Mantener ceros a la izquierda
denue_clean["cve_ent"] = denue_clean["cve_ent"].astype("string").str.strip().str.zfill(2)
denue_clean["cve_mun"] = denue_clean["cve_mun"].astype("string").str.strip().str.zfill(3)
denue_clean["cve_loc"] = denue_clean["cve_loc"].astype("string").str.strip().str.zfill(4)

# Llave municipal compatible con CVEGEO municipal
denue_clean["geo_key"] = denue_clean["cve_ent"] + denue_clean["cve_mun"]

# Llave legible, similar a la que has usado en otros ejercicios
denue_clean["geo_key_dash"] = denue_clean["cve_ent"] + "-" + denue_clean["cve_mun"]


# ------------------------------------------------------------
# 9. Crear alias consistentes con las bases de Censos / SAIC
# ------------------------------------------------------------

denue_clean["entidad_id"] = denue_clean["cve_ent"]
denue_clean["entidad_name"] = denue_clean["entidad"]

denue_clean["municipio_id"] = denue_clean["cve_mun"]
denue_clean["municipio_name"] = denue_clean["municipio"]

denue_clean["scian_code"] = denue_clean["codigo_act"]
denue_clean["scian_description"] = denue_clean["nombre_act"]


# ------------------------------------------------------------
# 10. Limpieza y creación de variables SCIAN
# ------------------------------------------------------------

denue_clean["year"] = pd.to_numeric(
    denue_clean["year"],
    errors="coerce"
).astype("Int64")

denue_clean["codigo_act"] = denue_clean["codigo_act"].astype("string").str.strip()
denue_clean["scian_code"] = denue_clean["scian_code"].astype("string").str.strip()

# Niveles SCIAN derivados del código de actividad
denue_clean["scian_sector"] = denue_clean["scian_code"].str.slice(0, 2)
denue_clean["scian_subsector"] = denue_clean["scian_code"].str.slice(0, 3)
denue_clean["scian_industry_group"] = denue_clean["scian_code"].str.slice(0, 4)
denue_clean["scian_industry"] = denue_clean["scian_code"].str.slice(0, 5)
denue_clean["scian_class"] = denue_clean["scian_code"].str.slice(0, 6)


# ------------------------------------------------------------
# 11. Convertir coordenadas a numéricas
# ------------------------------------------------------------

denue_clean["latitud"] = pd.to_numeric(denue_clean["latitud"], errors="coerce")
denue_clean["longitud"] = pd.to_numeric(denue_clean["longitud"], errors="coerce")


# ------------------------------------------------------------
# 12. Limpieza básica de columnas de texto
# ------------------------------------------------------------

text_columns = [
    "nom_estab",
    "raz_social",
    "nombre_act",
    "scian_description",
    "entidad",
    "entidad_name",
    "municipio",
    "municipio_name",
    "localidad",
    "tipo_vial",
    "nom_vial",
    "tipo_asent",
    "nomb_asent",
    "per_ocu",
    "tipoUniEco",
    "correoelec",
    "www"
]

for col in text_columns:
    if col in denue_clean.columns:
        denue_clean[col] = clean_text_column(denue_clean[col])


# ------------------------------------------------------------
# 13. Reordenar columnas finales
# ------------------------------------------------------------

final_column_order = [
    # Metadatos
    "year",
    "denue_sector_block",
    "denue_sector_label",
    "denue_sector_short_label",
    "source_file",

    # Identificación del establecimiento
    "id",
    "clee",
    "nom_estab",
    "raz_social",
    "tipoUniEco",

    # Actividad económica
    "codigo_act",
    "nombre_act",
    "scian_code",
    "scian_description",
    "scian_sector",
    "scian_subsector",
    "scian_industry_group",
    "scian_industry",
    "scian_class",
    "per_ocu",

    # Geografía original DENUE
    "cve_ent",
    "entidad",
    "cve_mun",
    "municipio",

    # Geografía homologada con Censos / SAIC
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    "geo_key_dash",

    # Geografía más granular
    "cve_loc",
    "localidad",
    "ageb",
    "manzana",
    "latitud",
    "longitud",

    # Dirección
    "tipo_vial",
    "nom_vial",
    "numero_ext",
    "letra_ext",
    "numero_int",
    "letra_int",
    "tipo_asent",
    "nomb_asent",
    "cod_postal",

    # Contacto
    "telefono",
    "correoelec",
    "www",

    # Fecha
    "fecha_alta"
]

final_column_order_existing = [
    col for col in final_column_order
    if col in denue_clean.columns
]

denue_clean = denue_clean[final_column_order_existing].copy()


# ------------------------------------------------------------
# 14. Validaciones rápidas
# ------------------------------------------------------------

print("\nVALIDACIÓN DE BASE DENUE LIMPIA")
print("=" * 100)

print(f"Filas totales: {denue_clean.shape[0]:,}")
print(f"Columnas totales: {denue_clean.shape[1]:,}")

print("\nFilas por año:")
display(
    denue_clean
    .groupby("year", dropna=False)
    .size()
    .reset_index(name="n_establishments")
)

print("\nFilas por año y bloque sectorial:")
display(
    denue_clean
    .groupby(
        ["year", "denue_sector_block", "denue_sector_label"],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "denue_sector_block"])
)

print("\nMunicipios únicos por año:")
display(
    denue_clean
    .groupby("year", dropna=False)["geo_key"]
    .nunique()
    .reset_index(name="n_municipalities")
)

print("\nValores faltantes en columnas clave:")

key_columns = [
    "year",
    "id",
    "codigo_act",
    "scian_code",
    "nombre_act",
    "scian_description",
    "cve_ent",
    "cve_mun",
    "geo_key",
    "entidad",
    "municipio",
    "latitud",
    "longitud"
]

missing_summary = (
    denue_clean[key_columns]
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_values"]
missing_summary["missing_share"] = (
    missing_summary["missing_values"] / len(denue_clean)
)

display(missing_summary)


# ------------------------------------------------------------
# 15. Vista rápida de la base final
# ------------------------------------------------------------

print("\nMUESTRA DE LA BASE FINAL")
print("=" * 100)

display(denue_clean.head(20))


# ------------------------------------------------------------
# 16. Guardar base limpia principal
# ------------------------------------------------------------

output_base_path = processed_path / "denue_industrial_support_base.csv"

denue_clean.to_csv(
    output_base_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nBASE PRINCIPAL GUARDADA EN:")
print(output_base_path)


# ------------------------------------------------------------
# 17. Crear reportes auxiliares
# ------------------------------------------------------------

# Reporte 1: conteo por año y bloque sectorial
denue_counts_by_year_sector = (
    denue_clean
    .groupby(
        ["year", "denue_sector_block", "denue_sector_label"],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "denue_sector_block"])
)

denue_counts_by_year_sector_path = (
    tables_path / "denue_counts_by_year_sector.csv"
)

denue_counts_by_year_sector.to_csv(
    denue_counts_by_year_sector_path,
    index=False,
    encoding="utf-8-sig"
)


# Reporte 2: conteo por año, entidad y bloque sectorial
denue_counts_by_state_sector = (
    denue_clean
    .groupby(
        [
            "year",
            "entidad_id",
            "entidad_name",
            "denue_sector_block",
            "denue_sector_label"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "entidad_id", "denue_sector_block"])
)

denue_counts_by_state_sector_path = (
    tables_path / "denue_counts_by_state_sector.csv"
)

denue_counts_by_state_sector.to_csv(
    denue_counts_by_state_sector_path,
    index=False,
    encoding="utf-8-sig"
)


# Reporte 3: conteo por año, municipio y bloque sectorial
denue_counts_by_municipality_sector = (
    denue_clean
    .groupby(
        [
            "year",
            "entidad_id",
            "entidad_name",
            "municipio_id",
            "municipio_name",
            "geo_key",
            "geo_key_dash",
            "denue_sector_block",
            "denue_sector_label"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "geo_key", "denue_sector_block"])
)

denue_counts_by_municipality_sector_path = (
    tables_path / "denue_counts_by_municipality_sector.csv"
)

denue_counts_by_municipality_sector.to_csv(
    denue_counts_by_municipality_sector_path,
    index=False,
    encoding="utf-8-sig"
)


# Reporte 4: catálogo de actividades SCIAN observadas
denue_scian_catalog = (
    denue_clean
    .groupby(
        [
            "year",
            "denue_sector_block",
            "denue_sector_label",
            "scian_code",
            "scian_description"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "denue_sector_block", "scian_code"])
)

denue_scian_catalog_path = (
    tables_path / "denue_scian_catalog_by_year.csv"
)

denue_scian_catalog.to_csv(
    denue_scian_catalog_path,
    index=False,
    encoding="utf-8-sig"
)


# Reporte 5: conteo por tamaño de establecimiento
denue_counts_by_size = (
    denue_clean
    .groupby(
        [
            "year",
            "denue_sector_block",
            "denue_sector_label",
            "per_ocu"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
    .sort_values(["year", "denue_sector_block", "per_ocu"])
)

denue_counts_by_size_path = (
    tables_path / "denue_counts_by_size.csv"
)

denue_counts_by_size.to_csv(
    denue_counts_by_size_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 18. Mensaje final
# ------------------------------------------------------------

print("\nREPORTES GUARDADOS EN:")
print(denue_counts_by_year_sector_path)
print(denue_counts_by_state_sector_path)
print(denue_counts_by_municipality_sector_path)
print(denue_scian_catalog_path)
print(denue_counts_by_size_path)

print("\nPROCESO TERMINADO")
print("=" * 100)

print("Base principal creada:")
print(output_base_path)

print("\nUnidad de observación:")
print("year × establishment")

print("\nColumnas clave para cruces futuros:")
print("year, geo_key, entidad_id, municipio_id, scian_code, denue_sector_block")

print("\nNota metodológica:")
print("La base DENUE queda disponible para 2018 y 2023.")
print("Para cruzarla con Censos/SAIC, filtra el panel censal a esos mismos años.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LECTURA DE ARCHIVOS DENUE
Archivo leído con encoding utf-8: denue_31_33_2023.csv
Filas: 611,331 | Columnas: 47
----------------------------------------------------------------------------------------------------
Archivo leído con encoding latin1: denue_48_49_2023.csv
Filas: 40,447 | Columnas: 47
----------------------------------------------------------------------------------------------------
Archivo leído con encoding latin1: denue_54_2023.csv
Filas: 108,192 | Columnas: 47
----------------------------------------------------------------------------------------------------
Archivo leído con encoding utf-8: denue_56_2023.csv
Filas: 80,268 | Columnas: 47
----------------------------------------------------------------------------------------------------
Archivo leído con encoding latin1: denue_31_33_2018.csv
Filas: 529,993 | Columnas: 47
---------------------

,year,n_establishments
0,2018,778329
1,2023,840238



Filas por año y bloque sectorial:


,year,denue_sector_block,denue_sector_label,n_establishments
0,2018,31_33,Industrias manufactureras,529993
1,2018,48_49,"Transportes, correos y almacenamiento",38383
2,2018,54,"Servicios profesionales, científicos y técnicos",103578
3,2018,56,Servicios de apoyo a los negocios y manejo de ...,106375
4,2023,31_33,Industrias manufactureras,611331
5,2023,48_49,"Transportes, correos y almacenamiento",40447
6,2023,54,"Servicios profesionales, científicos y técnicos",108192
7,2023,56,Servicios de apoyo a los negocios y manejo de ...,80268



Municipios únicos por año:


...


Valores faltantes en columnas clave:


,column,missing_values,missing_share
0,year,0,0.0
1,id,0,0.0
2,codigo_act,0,0.0
3,scian_code,0,0.0
4,nombre_act,0,0.0
5,scian_description,0,0.0
6,cve_ent,0,0.0
7,cve_mun,0,0.0
8,geo_key,0,0.0
9,entidad,0,0.0



MUESTRA DE LA BASE FINAL


,year,denue_sector_block,denue_sector_label,denue_sector_short_label,source_file,id,clee,nom_estab,raz_social,tipoUniEco,...,letra_ext,numero_int,letra_int,tipo_asent,nomb_asent,cod_postal,telefono,correoelec,www,fecha_alta
0,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7239103,01003311993000011000000000U0,PREPARACION Y VENTA DE SALSAS Y FRIJOLES,<NA>,Fijo,...,SN,0,NaN,COLONIA,LOS ANGELES,20800,NaN,<NA>,<NA>,2019-11
1,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,1079,01003312112000042000000000U1,PROCESADORA DE AGUA BIOS,AGUA PURIFICADA BIOS,Fijo,...,SN,NaN,NaN,LOCALIDAD,OJOCALIENTE,20834,NaN,<NA>,<NA>,2010-07
2,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7462448,01005311993000031000000000U3,POSTRES JESUS MARIA,<NA>,Fijo,...,SN,NaN,NaN,COLONIA,OJOS DE AGUA,20927,NaN,<NA>,<NA>,2019-11
3,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,13330,01001311613000041000000000U5,PREPARACION DE EMBUTIDOS Y OTRAS CONSERVAS,<NA>,Fijo,...,E,NaN,NaN,FRACCIONAMIENTO,JARDINES DE LA CONVENCION,20267,NaN,VILLA-H@LIVE.COM.MX,<NA>,2010-07
4,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,8387930,01005311910000124000000000U0,PROCESADORA DE BOTANAS Y ENCURTIDOS DE AGUASCA...,PROCESADORA DE BOTANAS Y ENCURTIDOS DE AGUASCA...,Fijo,...,NaN,NaN,NaN,PARQUE INDUSTRIAL,SAN ANTONIO DE LOS HORCONES,20916,NaN,PROCESADORA2009@HOTMAIL.COM,<NA>,2019-11
5,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7481597,01001312112002111000000000U6,PLANTA PURIFICADORA ENSOAB,<NA>,Fijo,...,NaN,NaN,NaN,PUEBLO,GRAL. JOSE MARIA MORELOS Y PAVON,20320,NaN,ENSO.FACTURACION@GMAIL.COM,<NA>,2019-11
6,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,6189344,01001315192000133000000000U1,PLAY BABY,<NA>,Fijo,...,B,NaN,NaN,FRACCIONAMIENTO,EL CAMINERO,20270,4491455970,CREACIONEPLAYBABY@HOTMAIL.COM,<NA>,2010-07
7,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,6281880,01001311110000153001004866S5,PLANTA DE ALIMENTOS DIVISION OCCIDENTE,BACHOCO SA DE CV,Fijo,...,S/N,NaN,NaN,COLONIA,SANTA MARIA DE GALLARDO,20324,NaN,RAFAEL.CERVANTES@BACHOCO.NET,BACHOCO.COM.MX,2014-12
8,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,8984744,01001315229002617000000000U1,PLANTA INISA SUCURSAL LAVAMEZ,INDUSTRIA DEL INTERIOR S DE RL,Fijo,...,SN,NaN,NaN,PARQUE INDUSTRIAL,SIGLO XXI,20290,NaN,<NA>,<NA>,2019-11
9,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,8540878,01001315223001442000000000U8,PLAY BALL,<NA>,Fijo,...,NaN,18,NaN,FRACCIONAMIENTO,SIDUSA,20260,4493291627,PLAY_BALL12@HOTMAIL.COM,<NA>,2019-11



BASE PRINCIPAL GUARDADA EN:
/content/drive/MyDrive/Nearshoring_Project/data/processed/denue_industrial_support_base.csv

REPORTES GUARDADOS EN:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_counts_by_year_sector.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_counts_by_state_sector.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_counts_by_municipality_sector.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_scian_catalog_by_year.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_counts_by_size.csv

PROCESO TERMINADO
Base principal creada:
/content/drive/MyDrive/Nearshoring_Project/data/processed/denue_industrial_support_base.csv

Unidad de observación:
year × establishment

Columnas clave para cruces futuros:
year, geo_key, entidad_id, municipio_id, scian_code, denue_sector_block

Nota metodológica:
La base DENUE queda disponible para 2018 y 2023.
Para cruzarla con Censos/SAIC, filtra el panel

In [ ]:
# ============================================================
# CREAR MUESTRA PEQUEÑA DE DENUE CON TODAS LAS COLUMNAS
# Proyecto: Nearshoring_Project
# Objetivo: visualizar la estructura completa en Google Sheets
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Definir rutas
# ------------------------------------------------------------

input_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/data/processed/denue_industrial_support_base.csv"
)

output_folder = Path(
    "/content/drive/MyDrive/Nearshoring_Project/outputs/tables"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_sample_path = output_folder / "denue_sample_all_columns_for_google_sheets.csv"


# ------------------------------------------------------------
# 2. Parámetros
# ------------------------------------------------------------

sample_size = 1000        # Puedes cambiarlo a 500, 2000, etc.
chunk_size = 100_000      # Lee la base por bloques para no saturar memoria
random_state = 123


# ------------------------------------------------------------
# 3. Leer por bloques y muestrear SIN quitar columnas
# ------------------------------------------------------------

sample_chunks = []

print("Leyendo base por bloques y tomando muestra con TODAS las columnas...")
print("=" * 100)

for i, chunk in enumerate(
    pd.read_csv(
        input_path,
        dtype=str,
        chunksize=chunk_size,
        low_memory=False
    )
):
    # Tomamos una pequeña muestra de cada bloque
    n_chunk_sample = max(1, int(sample_size / 20))
    n_to_sample = min(n_chunk_sample, len(chunk))

    chunk_sample = chunk.sample(
        n=n_to_sample,
        random_state=random_state + i
    )

    sample_chunks.append(chunk_sample)

    print(
        f"Chunk {i+1}: {len(chunk):,} filas leídas | "
        f"{n_to_sample:,} filas muestreadas | "
        f"{chunk.shape[1]} columnas"
    )


# ------------------------------------------------------------
# 4. Unir muestras parciales
# ------------------------------------------------------------

denue_sample = pd.concat(sample_chunks, ignore_index=True)

# Si quedó mayor al tamaño deseado, reducimos a sample_size
if len(denue_sample) > sample_size:
    denue_sample = denue_sample.sample(
        n=sample_size,
        random_state=random_state
    ).reset_index(drop=True)

print("\nMuestra final creada")
print("=" * 100)
print(f"Filas en muestra final: {len(denue_sample):,}")
print(f"Columnas en muestra final: {denue_sample.shape[1]:,}")


# ------------------------------------------------------------
# 5. Visualizar muestra en Colab
# ------------------------------------------------------------

display(denue_sample.head(30))


# ------------------------------------------------------------
# 6. Mostrar listado completo de columnas
# ------------------------------------------------------------

print("\nColumnas incluidas en la muestra:")
print("=" * 100)

for i, col in enumerate(denue_sample.columns, start=1):
    print(f"{i}. {col}")


# ------------------------------------------------------------
# 7. Guardar muestra para abrir en Google Sheets
# ------------------------------------------------------------

denue_sample.to_csv(
    output_sample_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo guardado en:")
print(output_sample_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Leyendo base por bloques y tomando muestra con TODAS las columnas...
Chunk 1: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 2: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 3: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 4: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 5: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 6: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 7: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 8: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 9: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 10: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 11: 100,000 filas leídas | 50 filas muestreadas | 49 columnas
Chunk 12: 100,000 filas leídas | 50 filas muestreadas | 49 colu

,year,denue_sector_block,denue_sector_label,denue_sector_short_label,source_file,id,clee,nom_estab,raz_social,tipoUniEco,...,letra_ext,numero_int,letra_int,tipo_asent,nomb_asent,cod_postal,telefono,correoelec,www,fecha_alta
0,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7131087,07059311812000771000000000U7,PANADERIA SIN NOMBRE,NaN,Fijo,...,SN,NaN,NaN,BARRIO,CINTALAPA,29954,NaN,NaN,NaN,2019-11
1,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,831955,09007315229000351000000000U8,MAQUILA CHAVEZ,NaN,Fijo,...,MANZANA 17,NaN,NaN,COLONIA,PUENTE BLANCO,09770,NaN,NaN,NaN,2010-07
2,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,9031980,09016332320000991000000000U8,HERRERIA JUAREZ,NaN,Fijo,...,A,NaN,NaN,COLONIA,OBSERVARIO,11840,5537290091,NaN,NaN,2019-11
3,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,498182,07046311830000191000000000U7,TORTILLERIA BELEN,NaN,Fijo,...,SN,NaN,NaN,BARRIO,SANTA CRUZ,30430,NaN,NaN,NaN,2010-07
4,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7346623,07086315225000111000000000U0,CONFECCIONES PALOMA,NaN,Fijo,...,SN,NaN,NaN,BARRIO,ABSALON CASTELLANOS,29150,9613487780,MODASPALOMA@GMAIL.COM,NaN,2019-11
5,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,493510,07108311830000421000000000U6,TORTILLERIA ANA ISABEL,NaN,Fijo,...,SN,NaN,NaN,AMPLIACION,SAN MIGUEL,30470,NaN,NaN,NaN,2010-07
6,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,525084,08059311812000051000000000U3,PANADERIA JUAREZ,NaN,Fijo,...,NaN,0,NaN,BARRIO,SAN FRANCISCO DEL ORO,33500,6271031649,NaN,NaN,2010-07
7,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,7651442,07122311830000021000000000U5,TORTILLERIA LLAVEN,NaN,Fijo,...,SN,NaN,NaN,BARRIO,LAS CASITAS,30530,NaN,NaN,NaN,2019-11
8,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,502472,07054332320000021000000000U7,BALCONERIA SIN NOMBRE,NaN,Fijo,...,SN,NaN,NaN,BARRIO,SANTA TERESA,30650,9622251603,NaN,NaN,2010-07
9,2023,31_33,Industrias manufactureras,manufacturing,denue_31_33_2023.csv,46913,01009332320000121000000000U5,BALCONERIA ROMAN,NaN,Fijo,...,NaN,NaN,NaN,COLONIA,CENTRO,20616,4651089617,NaN,NaN,2014-12



Columnas incluidas en la muestra:
1. year
2. denue_sector_block
3. denue_sector_label
4. denue_sector_short_label
5. source_file
6. id
7. clee
8. nom_estab
9. raz_social
10. tipoUniEco
11. codigo_act
12. nombre_act
13. scian_code
14. scian_description
15. scian_sector
16. scian_subsector
17. scian_industry_group
18. scian_industry
19. scian_class
20. per_ocu
21. cve_ent
22. entidad
23. cve_mun
24. municipio
25. entidad_id
26. entidad_name
27. municipio_id
28. municipio_name
29. geo_key
30. geo_key_dash
31. cve_loc
32. localidad
33. ageb
34. manzana
35. latitud
36. longitud
37. tipo_vial
38. nom_vial
39. numero_ext
40. letra_ext
41. numero_int
42. letra_int
43. tipo_asent
44. nomb_asent
45. cod_postal
46. telefono
47. correoelec
48. www
49. fecha_alta

Archivo guardado en:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_sample_all_columns_for_google_sheets.csv


In [2]:
# ============================================================
# CREAR BASE MUNICIPAL DENUE PARA CRUZAR CON SAIC
# Proyecto: Nearshoring_Project
#
# Input:
# data/processed/denue_industrial_support_base.csv
#
# Outputs:
# 1. data/processed/municipality_denue_support_summary.csv
# 2. outputs/tables/municipality_denue_support_summary_for_google_sheets.csv
# 3. outputs/tables/municipality_denue_support_by_sector_size_long.csv
#
# Unidad principal:
# year × municipality
#
# Lógica:
# - SAIC/Censos: capacidad manufacturera municipal
# - DENUE 48-49, 54, 56: oferta local de servicios B2B de soporte
# - DENUE 31-33: presencia manufacturera complementaria, no oferta B2B
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 1. Librerías
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 2. Rutas
# ------------------------------------------------------------

input_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/data/processed/denue_industrial_support_base.csv"
)

processed_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/data/processed"
)

tables_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/outputs/tables"
)

processed_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Leer solo columnas necesarias
# ------------------------------------------------------------
# Para no cargar columnas de dirección/contacto que no necesitamos
# en el agregado municipal.

columns_to_read = [
    "year",
    "id",
    "denue_sector_block",
    "denue_sector_label",
    "denue_sector_short_label",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    "scian_code",
    "scian_description",
    "scian_class",
    "per_ocu"
]

denue = pd.read_csv(
    input_path,
    dtype=str,
    usecols=lambda col: col in columns_to_read,
    low_memory=False
)

print("Base DENUE cargada")
print("=" * 100)
print(f"Filas: {denue.shape[0]:,}")
print(f"Columnas: {denue.shape[1]:,}")
display(denue.head())


# ------------------------------------------------------------
# 4. Limpieza mínima de claves y tipos
# ------------------------------------------------------------

denue["year"] = pd.to_numeric(
    denue["year"],
    errors="coerce"
).astype("Int64")

denue["entidad_id"] = (
    denue["entidad_id"]
    .astype("string")
    .str.strip()
    .str.zfill(2)
)

denue["municipio_id"] = (
    denue["municipio_id"]
    .astype("string")
    .str.strip()
    .str.zfill(3)
)

denue["geo_key"] = denue["entidad_id"] + denue["municipio_id"]

denue["scian_code"] = (
    denue["scian_code"]
    .astype("string")
    .str.strip()
)

# Si scian_class no existiera o viniera vacía, la reconstruimos con scian_code
if "scian_class" not in denue.columns:
    denue["scian_class"] = denue["scian_code"].str.slice(0, 6)
else:
    denue["scian_class"] = denue["scian_class"].fillna(
        denue["scian_code"].str.slice(0, 6)
    )

# Limpieza básica de texto
for col in ["entidad_name", "municipio_name", "denue_sector_label", "per_ocu"]:
    denue[col] = (
        denue[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


# ------------------------------------------------------------
# 5. Clasificación de tamaño usando per_ocu
# ------------------------------------------------------------
# Propuesta definida:
# micro  = 0 a 5 personas
# small  = 6 a 30 personas
# medium = 31 a 100 personas
# large  = 101 o más personas

def classify_business_size(per_ocu):
    """
    Clasifica el estrato de personal ocupado de DENUE en 4 grupos:
    micro, small, medium, large.
    """

    if pd.isna(per_ocu):
        return "unknown"

    value = str(per_ocu).strip().lower()

    if "0 a 5" in value:
        return "micro"

    elif "6 a 10" in value or "11 a 30" in value:
        return "small"

    elif "31 a 50" in value or "51 a 100" in value:
        return "medium"

    elif "101 a 250" in value or "251" in value:
        return "large"

    else:
        return "unknown"


denue["business_size_group"] = denue["per_ocu"].apply(classify_business_size)

print("\nDistribución nacional por tamaño de establecimiento")
print("=" * 100)
display(
    denue["business_size_group"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "business_size_group", "business_size_group": "n_establishments"})
)


# ------------------------------------------------------------
# 6. Identificar bloques analíticos
# ------------------------------------------------------------

# 31-33: manufactura DENUE, variable auxiliar
denue["is_manufacturing_denue"] = (
    denue["denue_sector_block"] == "31_33"
).astype(int)

# 48-49: transporte, correos y almacenamiento
denue["is_logistics_storage"] = (
    denue["denue_sector_block"] == "48_49"
).astype(int)

# 54: servicios profesionales, científicos y técnicos
denue["is_professional_technical"] = (
    denue["denue_sector_block"] == "54"
).astype(int)

# 56: apoyo a negocios, desechos y remediación
denue["is_business_support"] = (
    denue["denue_sector_block"] == "56"
).astype(int)

# Oferta B2B central: excluye manufactura
denue["is_b2b_support_service"] = (
    denue["denue_sector_block"].isin(["48_49", "54", "56"])
).astype(int)


# ------------------------------------------------------------
# 7. Crear variables binarias por tamaño
# ------------------------------------------------------------

denue["is_micro"] = (denue["business_size_group"] == "micro").astype(int)
denue["is_small"] = (denue["business_size_group"] == "small").astype(int)
denue["is_medium"] = (denue["business_size_group"] == "medium").astype(int)
denue["is_large"] = (denue["business_size_group"] == "large").astype(int)

denue["is_medium_large"] = (
    denue["business_size_group"].isin(["medium", "large"])
).astype(int)


# ------------------------------------------------------------
# 8. Crear variables B2B por tamaño
# ------------------------------------------------------------

denue["is_b2b_micro"] = (
    (denue["is_b2b_support_service"] == 1) &
    (denue["business_size_group"] == "micro")
).astype(int)

denue["is_b2b_small"] = (
    (denue["is_b2b_support_service"] == 1) &
    (denue["business_size_group"] == "small")
).astype(int)

denue["is_b2b_medium"] = (
    (denue["is_b2b_support_service"] == 1) &
    (denue["business_size_group"] == "medium")
).astype(int)

denue["is_b2b_large"] = (
    (denue["is_b2b_support_service"] == 1) &
    (denue["business_size_group"] == "large")
).astype(int)

denue["is_b2b_medium_large"] = (
    (denue["is_b2b_support_service"] == 1) &
    (denue["business_size_group"].isin(["medium", "large"]))
).astype(int)


# ------------------------------------------------------------
# 9. Agregar a nivel municipal
# ------------------------------------------------------------

group_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key"
]

municipality_denue_summary = (
    denue
    .groupby(group_cols, dropna=False)
    .agg(
        # Conteo total de establecimientos en los bloques descargados
        denue_total_establishments=("id", "count"),

        # Bloque manufacturero DENUE: complemento / validación
        denue_manufacturing_establishments=("is_manufacturing_denue", "sum"),

        # Bloques de oferta B2B
        denue_logistics_storage_establishments=("is_logistics_storage", "sum"),
        denue_professional_technical_establishments=("is_professional_technical", "sum"),
        denue_business_support_establishments=("is_business_support", "sum"),

        # Total B2B: 48-49 + 54 + 56
        denue_b2b_support_establishments=("is_b2b_support_service", "sum"),

        # Diversidad SCIAN total
        denue_total_scian_classes=("scian_class", "nunique"),

        # Conteos por tamaño, todos los bloques
        denue_micro_establishments=("is_micro", "sum"),
        denue_small_establishments=("is_small", "sum"),
        denue_medium_establishments=("is_medium", "sum"),
        denue_large_establishments=("is_large", "sum"),
        denue_medium_large_establishments=("is_medium_large", "sum"),

        # Conteos por tamaño, solo B2B
        denue_b2b_micro_establishments=("is_b2b_micro", "sum"),
        denue_b2b_small_establishments=("is_b2b_small", "sum"),
        denue_b2b_medium_establishments=("is_b2b_medium", "sum"),
        denue_b2b_large_establishments=("is_b2b_large", "sum"),
        denue_b2b_medium_large_establishments=("is_b2b_medium_large", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 10. Diversidad SCIAN B2B por municipio
# ------------------------------------------------------------
# Se calcula aparte para contar clases SCIAN solo dentro de B2B.

b2b_scian_diversity = (
    denue[denue["is_b2b_support_service"] == 1]
    .groupby(group_cols, dropna=False)["scian_class"]
    .nunique()
    .reset_index(name="denue_b2b_support_scian_classes")
)

municipality_denue_summary = municipality_denue_summary.merge(
    b2b_scian_diversity,
    on=group_cols,
    how="left"
)

municipality_denue_summary["denue_b2b_support_scian_classes"] = (
    municipality_denue_summary["denue_b2b_support_scian_classes"]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 11. Shares por tamaño de establecimiento
# ------------------------------------------------------------

def safe_share(numerator, denominator):
    """
    Calcula participaciones evitando división entre cero.
    """
    return numerator / denominator.replace(0, np.nan)


# Shares por tamaño sobre el total de establecimientos descargados
municipality_denue_summary["share_micro_establishments"] = safe_share(
    municipality_denue_summary["denue_micro_establishments"],
    municipality_denue_summary["denue_total_establishments"]
)

municipality_denue_summary["share_small_establishments"] = safe_share(
    municipality_denue_summary["denue_small_establishments"],
    municipality_denue_summary["denue_total_establishments"]
)

municipality_denue_summary["share_medium_establishments"] = safe_share(
    municipality_denue_summary["denue_medium_establishments"],
    municipality_denue_summary["denue_total_establishments"]
)

municipality_denue_summary["share_large_establishments"] = safe_share(
    municipality_denue_summary["denue_large_establishments"],
    municipality_denue_summary["denue_total_establishments"]
)

municipality_denue_summary["share_medium_large_establishments"] = safe_share(
    municipality_denue_summary["denue_medium_large_establishments"],
    municipality_denue_summary["denue_total_establishments"]
)


# Shares por tamaño sobre el total B2B
municipality_denue_summary["share_b2b_micro_establishments"] = safe_share(
    municipality_denue_summary["denue_b2b_micro_establishments"],
    municipality_denue_summary["denue_b2b_support_establishments"]
)

municipality_denue_summary["share_b2b_small_establishments"] = safe_share(
    municipality_denue_summary["denue_b2b_small_establishments"],
    municipality_denue_summary["denue_b2b_support_establishments"]
)

municipality_denue_summary["share_b2b_medium_establishments"] = safe_share(
    municipality_denue_summary["denue_b2b_medium_establishments"],
    municipality_denue_summary["denue_b2b_support_establishments"]
)

municipality_denue_summary["share_b2b_large_establishments"] = safe_share(
    municipality_denue_summary["denue_b2b_large_establishments"],
    municipality_denue_summary["denue_b2b_support_establishments"]
)

municipality_denue_summary["share_b2b_medium_large_establishments"] = safe_share(
    municipality_denue_summary["denue_b2b_medium_large_establishments"],
    municipality_denue_summary["denue_b2b_support_establishments"]
)


# Reemplazar NA generados por municipios sin B2B con cero
share_cols = [
    "share_micro_establishments",
    "share_small_establishments",
    "share_medium_establishments",
    "share_large_establishments",
    "share_medium_large_establishments",
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments"
]

municipality_denue_summary[share_cols] = (
    municipality_denue_summary[share_cols]
    .fillna(0)
)


# ------------------------------------------------------------
# 12. Crear tabla larga por sector y tamaño
# ------------------------------------------------------------
# Esta tabla es útil para estadística descriptiva y visualizaciones.

municipality_denue_sector_size_long = (
    denue
    .groupby(
        group_cols + [
            "denue_sector_block",
            "denue_sector_label",
            "business_size_group"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_establishments")
)

# Total por municipio-sector para calcular share dentro de cada sector
sector_totals = (
    municipality_denue_sector_size_long
    .groupby(
        group_cols + [
            "denue_sector_block",
            "denue_sector_label"
        ],
        dropna=False
    )["n_establishments"]
    .sum()
    .reset_index(name="sector_municipality_total")
)

municipality_denue_sector_size_long = municipality_denue_sector_size_long.merge(
    sector_totals,
    on=group_cols + ["denue_sector_block", "denue_sector_label"],
    how="left"
)

municipality_denue_sector_size_long["share_within_sector_municipality"] = (
    municipality_denue_sector_size_long["n_establishments"] /
    municipality_denue_sector_size_long["sector_municipality_total"].replace(0, np.nan)
)

municipality_denue_sector_size_long["share_within_sector_municipality"] = (
    municipality_denue_sector_size_long["share_within_sector_municipality"]
    .fillna(0)
)


# ------------------------------------------------------------
# 13. Ordenar columnas de la base municipal principal
# ------------------------------------------------------------

final_columns = [
    # Llaves
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    # Totales generales
    "denue_total_establishments",
    "denue_total_scian_classes",

    # Manufactura DENUE como complemento
    "denue_manufacturing_establishments",

    # Oferta B2B por bloque
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_support_scian_classes",

    # Tamaño, todos los bloques
    "denue_micro_establishments",
    "denue_small_establishments",
    "denue_medium_establishments",
    "denue_large_establishments",
    "denue_medium_large_establishments",

    # Shares tamaño, todos los bloques
    "share_micro_establishments",
    "share_small_establishments",
    "share_medium_establishments",
    "share_large_establishments",
    "share_medium_large_establishments",

    # Tamaño, solo B2B
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    # Shares tamaño, solo B2B
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments"
]

municipality_denue_summary = municipality_denue_summary[final_columns].copy()

municipality_denue_summary = municipality_denue_summary.sort_values(
    ["year", "geo_key"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 14. Validaciones rápidas
# ------------------------------------------------------------

print("\nBASE MUNICIPAL DENUE CREADA")
print("=" * 100)
print(f"Filas: {municipality_denue_summary.shape[0]:,}")
print(f"Columnas: {municipality_denue_summary.shape[1]:,}")

print("\nMunicipios por año:")
display(
    municipality_denue_summary
    .groupby("year")["geo_key"]
    .nunique()
    .reset_index(name="n_municipalities")
)

print("\nSuma nacional por año:")
display(
    municipality_denue_summary
    .groupby("year")
    [
        [
            "denue_total_establishments",
            "denue_manufacturing_establishments",
            "denue_b2b_support_establishments",
            "denue_logistics_storage_establishments",
            "denue_professional_technical_establishments",
            "denue_business_support_establishments",
            "denue_b2b_micro_establishments",
            "denue_b2b_small_establishments",
            "denue_b2b_medium_establishments",
            "denue_b2b_large_establishments"
        ]
    ]
    .sum()
    .reset_index()
)

print("\nMuestra de la base municipal:")
display(municipality_denue_summary.head(30))


# ------------------------------------------------------------
# 15. Estadística descriptiva inicial de shares B2B
# ------------------------------------------------------------

b2b_share_descriptive = (
    municipality_denue_summary
    [
        [
            "share_b2b_micro_establishments",
            "share_b2b_small_establishments",
            "share_b2b_medium_establishments",
            "share_b2b_large_establishments",
            "share_b2b_medium_large_establishments"
        ]
    ]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

print("\nEstadística descriptiva de shares B2B por tamaño:")
display(b2b_share_descriptive)


# ------------------------------------------------------------
# 16. Guardar outputs
# ------------------------------------------------------------

main_output_path = (
    processed_path / "municipality_denue_support_summary.csv"
)

sheets_output_path = (
    tables_path / "municipality_denue_support_summary_for_google_sheets.csv"
)

long_output_path = (
    tables_path / "municipality_denue_support_by_sector_size_long.csv"
)

descriptive_output_path = (
    tables_path / "denue_b2b_size_shares_descriptive_statistics.csv"
)

municipality_denue_summary.to_csv(
    main_output_path,
    index=False,
    encoding="utf-8-sig"
)

municipality_denue_summary.to_csv(
    sheets_output_path,
    index=False,
    encoding="utf-8-sig"
)

municipality_denue_sector_size_long.to_csv(
    long_output_path,
    index=False,
    encoding="utf-8-sig"
)

b2b_share_descriptive.to_csv(
    descriptive_output_path,
    index=False,
    encoding="utf-8-sig"
)


print("\nARCHIVOS GUARDADOS")
print("=" * 100)
print("Base municipal principal:")
print(main_output_path)

print("\nVersión para Google Sheets:")
print(sheets_output_path)

print("\nTabla larga por sector y tamaño:")
print(long_output_path)

print("\nEstadística descriptiva de shares B2B:")
print(descriptive_output_path)


# ------------------------------------------------------------
# 17. Nota final
# ------------------------------------------------------------

print("\nPROCESO TERMINADO")
print("=" * 100)
print("La base municipal está lista para cruzarse con SAIC/Censos usando:")
print("year + geo_key")

print("\nRecuerda:")
print("- denue_manufacturing_establishments es complemento/validación.")
print("- denue_b2b_support_establishments es la variable central de oferta B2B.")
print("- Los shares B2B por tamaño describen qué tan atomizada o escalada es la oferta local.")

Mounted at /content/drive
Base DENUE cargada
Filas: 1,618,567
Columnas: 14


,year,denue_sector_block,denue_sector_label,denue_sector_short_label,id,scian_code,scian_description,scian_class,per_ocu,entidad_id,entidad_name,municipio_id,municipio_name,geo_key
0,2023,31_33,Industrias manufactureras,manufacturing,7239103,311993,Elaboración de alimentos frescos para consumo ...,311993,0 a 5 personas,01,Aguascalientes,003,Calvillo,01003
1,2023,31_33,Industrias manufactureras,manufacturing,1079,312112,Purificación y embotellado de agua,312112,0 a 5 personas,01,Aguascalientes,003,Calvillo,01003
2,2023,31_33,Industrias manufactureras,manufacturing,7462448,311993,Elaboración de alimentos frescos para consumo ...,311993,0 a 5 personas,01,Aguascalientes,005,Jesús María,01005
3,2023,31_33,Industrias manufactureras,manufacturing,13330,311613,Preparación de embutidos y otras conservas de ...,311613,0 a 5 personas,01,Aguascalientes,001,Aguascalientes,01001
4,2023,31_33,Industrias manufactureras,manufacturing,8387930,311910,Elaboración de botanas,311910,31 a 50 personas,01,Aguascalientes,005,Jesús María,01005



Distribución nacional por tamaño de establecimiento


,n_establishments,count
0,micro,1365010
1,small,187720
2,medium,38008
3,large,27829



BASE MUNICIPAL DENUE CREADA
Filas: 4,905
Columnas: 34

Municipios por año:


...


Muestra de la base municipal:


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_total_establishments,denue_total_scian_classes,denue_manufacturing_establishments,denue_logistics_storage_establishments,...,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,share_b2b_micro_establishments,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments
0,2018,01,AGUASCALIENTES,001,Aguascalientes,01001,6740,281,3802,352,...,2262,536,99,41,140,0.769912,0.182437,0.033696,0.013955,0.047651
1,2018,01,AGUASCALIENTES,002,Asientos,01002,72,20,53,2,...,18,1,0,0,0,0.947368,0.052632,0.000000,0.000000,0.000000
2,2018,01,AGUASCALIENTES,003,Calvillo,01003,291,61,205,7,...,79,5,2,0,2,0.918605,0.058140,0.023256,0.000000,0.023256
3,2018,01,AGUASCALIENTES,004,Cosío,01004,44,16,31,1,...,12,1,0,0,0,0.923077,0.076923,0.000000,0.000000,0.000000
4,2018,01,AGUASCALIENTES,005,Jesús María,01005,854,160,651,58,...,146,40,14,3,17,0.719212,0.197044,0.068966,0.014778,0.083744
5,2018,01,AGUASCALIENTES,006,Pabellón de Arteaga,01006,208,41,144,3,...,58,5,1,0,1,0.906250,0.078125,0.015625,0.000000,0.015625
6,2018,01,AGUASCALIENTES,007,Rincón de Romos,01007,261,54,176,5,...,82,2,0,1,1,0.964706,0.023529,0.000000,0.011765,0.011765
7,2018,01,AGUASCALIENTES,008,San José de Gracia,01008,42,19,34,3,...,5,3,0,0,0,0.625000,0.375000,0.000000,0.000000,0.000000
8,2018,01,AGUASCALIENTES,009,Tepezalá,01009,49,18,38,0,...,10,0,0,1,1,0.909091,0.000000,0.000000,0.090909,0.090909
9,2018,01,AGUASCALIENTES,010,El Llano,01010,38,19,29,0,...,9,0,0,0,0,1.000000,0.000000,0.000000,0.000000,0.000000



Estadística descriptiva de shares B2B por tamaño:


,variable,count,mean,std,min,25%,50%,75%,max
0,share_b2b_micro_establishments,4905.0,0.839448,0.275426,0.0,0.832936,0.944444,1.000000,1.0
1,share_b2b_small_establishments,4905.0,0.064964,0.112911,0.0,0.000000,0.000000,0.100000,1.0
2,share_b2b_medium_establishments,4905.0,0.012801,0.038086,0.0,0.000000,0.000000,0.005263,1.0
3,share_b2b_large_establishments,4905.0,0.005723,0.033406,0.0,0.000000,0.000000,0.000000,1.0
4,share_b2b_medium_large_establishments,4905.0,0.018524,0.053940,0.0,0.000000,0.000000,0.013889,1.0



ARCHIVOS GUARDADOS
Base municipal principal:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_denue_support_summary.csv

Versión para Google Sheets:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/municipality_denue_support_summary_for_google_sheets.csv

Tabla larga por sector y tamaño:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/municipality_denue_support_by_sector_size_long.csv

Estadística descriptiva de shares B2B:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_b2b_size_shares_descriptive_statistics.csv

PROCESO TERMINADO
La base municipal está lista para cruzarse con SAIC/Censos usando:
year + geo_key

Recuerda:
- denue_manufacturing_establishments es complemento/validación.
- denue_b2b_support_establishments es la variable central de oferta B2B.
- Los shares B2B por tamaño describen qué tan atomizada o escalada es la oferta local.


In [3]:
# ============================================================
# VALIDACIÓN DE SERVICIOS B2B MEDIUM/LARGE EN DENUE MUNICIPAL
# Proyecto: Nearshoring_Project
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

input_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_denue_support_summary.csv"
)

tables_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/outputs/tables"
)

tables_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Cargar base municipal DENUE
# ------------------------------------------------------------

denue_mun = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

# Asegurar formato correcto de claves
denue_mun["year"] = pd.to_numeric(denue_mun["year"], errors="coerce").astype("Int64")
denue_mun["entidad_id"] = denue_mun["entidad_id"].astype(str).str.zfill(2)
denue_mun["municipio_id"] = denue_mun["municipio_id"].astype(str).str.zfill(3)
denue_mun["geo_key"] = denue_mun["entidad_id"] + denue_mun["municipio_id"]

print("Base cargada")
print("=" * 100)
print(f"Filas: {denue_mun.shape[0]:,}")
print(f"Columnas: {denue_mun.shape[1]:,}")
display(denue_mun.head())


# ------------------------------------------------------------
# 3. Crear banderas interpretativas
# ------------------------------------------------------------

denue_mun["has_b2b_support"] = (
    denue_mun["denue_b2b_support_establishments"] > 0
).astype(int)

denue_mun["has_b2b_medium_large"] = (
    denue_mun["denue_b2b_medium_large_establishments"] > 0
).astype(int)

denue_mun["has_b2b_large"] = (
    denue_mun["denue_b2b_large_establishments"] > 0
).astype(int)

denue_mun["has_b2b_only_micro_small"] = (
    (denue_mun["denue_b2b_support_establishments"] > 0) &
    (denue_mun["denue_b2b_medium_large_establishments"] == 0)
).astype(int)

denue_mun["has_no_b2b_support"] = (
    denue_mun["denue_b2b_support_establishments"] == 0
).astype(int)


# ------------------------------------------------------------
# 4. Tabla resumen por año
# ------------------------------------------------------------

validation_summary = (
    denue_mun
    .groupby("year", dropna=False)
    .agg(
        n_municipalities=("geo_key", "nunique"),

        municipalities_without_b2b=("has_no_b2b_support", "sum"),
        municipalities_with_b2b=("has_b2b_support", "sum"),
        municipalities_with_b2b_only_micro_small=("has_b2b_only_micro_small", "sum"),
        municipalities_with_b2b_medium_large=("has_b2b_medium_large", "sum"),
        municipalities_with_b2b_large=("has_b2b_large", "sum"),

        total_b2b_establishments=("denue_b2b_support_establishments", "sum"),
        total_b2b_micro=("denue_b2b_micro_establishments", "sum"),
        total_b2b_small=("denue_b2b_small_establishments", "sum"),
        total_b2b_medium=("denue_b2b_medium_establishments", "sum"),
        total_b2b_large=("denue_b2b_large_establishments", "sum"),
        total_b2b_medium_large=("denue_b2b_medium_large_establishments", "sum")
    )
    .reset_index()
)

# Shares municipales
validation_summary["share_municipalities_without_b2b"] = (
    validation_summary["municipalities_without_b2b"] /
    validation_summary["n_municipalities"]
)

validation_summary["share_municipalities_with_b2b"] = (
    validation_summary["municipalities_with_b2b"] /
    validation_summary["n_municipalities"]
)

validation_summary["share_municipalities_with_b2b_only_micro_small"] = (
    validation_summary["municipalities_with_b2b_only_micro_small"] /
    validation_summary["n_municipalities"]
)

validation_summary["share_municipalities_with_b2b_medium_large"] = (
    validation_summary["municipalities_with_b2b_medium_large"] /
    validation_summary["n_municipalities"]
)

validation_summary["share_municipalities_with_b2b_large"] = (
    validation_summary["municipalities_with_b2b_large"] /
    validation_summary["n_municipalities"]
)

# Shares nacionales de establecimientos B2B por tamaño
validation_summary["share_b2b_micro_national"] = (
    validation_summary["total_b2b_micro"] /
    validation_summary["total_b2b_establishments"].replace(0, np.nan)
)

validation_summary["share_b2b_small_national"] = (
    validation_summary["total_b2b_small"] /
    validation_summary["total_b2b_establishments"].replace(0, np.nan)
)

validation_summary["share_b2b_medium_national"] = (
    validation_summary["total_b2b_medium"] /
    validation_summary["total_b2b_establishments"].replace(0, np.nan)
)

validation_summary["share_b2b_large_national"] = (
    validation_summary["total_b2b_large"] /
    validation_summary["total_b2b_establishments"].replace(0, np.nan)
)

validation_summary["share_b2b_medium_large_national"] = (
    validation_summary["total_b2b_medium_large"] /
    validation_summary["total_b2b_establishments"].replace(0, np.nan)
)

print("\nRESUMEN DE VALIDACIÓN POR AÑO")
print("=" * 100)
display(validation_summary)


# ------------------------------------------------------------
# 5. Top municipios por B2B medium/large
# ------------------------------------------------------------

top_b2b_medium_large = (
    denue_mun
    .sort_values(
        ["year", "denue_b2b_medium_large_establishments"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    [
        [
            "year",
            "entidad_name",
            "municipio_name",
            "geo_key",
            "denue_b2b_support_establishments",
            "denue_b2b_medium_large_establishments",
            "denue_b2b_large_establishments",
            "share_b2b_medium_large_establishments",
            "share_b2b_large_establishments",
            "denue_logistics_storage_establishments",
            "denue_professional_technical_establishments",
            "denue_business_support_establishments"
        ]
    ]
)

print("\nTOP 30 MUNICIPIOS POR B2B MEDIUM/LARGE, POR AÑO")
print("=" * 100)
display(top_b2b_medium_large)


# ------------------------------------------------------------
# 6. Municipios con B2B pero sin medium/large
# ------------------------------------------------------------

b2b_only_micro_small = (
    denue_mun
    .query("denue_b2b_support_establishments > 0 and denue_b2b_medium_large_establishments == 0")
    .sort_values(
        ["year", "denue_b2b_support_establishments"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    [
        [
            "year",
            "entidad_name",
            "municipio_name",
            "geo_key",
            "denue_b2b_support_establishments",
            "denue_b2b_micro_establishments",
            "denue_b2b_small_establishments",
            "denue_b2b_medium_establishments",
            "denue_b2b_large_establishments",
            "share_b2b_micro_establishments",
            "share_b2b_small_establishments"
        ]
    ]
)

print("\nTOP MUNICIPIOS CON B2B PERO SOLO MICRO/SMALL")
print("=" * 100)
display(b2b_only_micro_small)


# ------------------------------------------------------------
# 7. Estadística descriptiva solo para municipios con B2B
# ------------------------------------------------------------

denue_mun_with_b2b = denue_mun[
    denue_mun["denue_b2b_support_establishments"] > 0
].copy()

descriptive_with_b2b = (
    denue_mun_with_b2b
    .groupby("year")
    [
        [
            "denue_b2b_support_establishments",
            "denue_b2b_medium_large_establishments",
            "denue_b2b_large_establishments",
            "share_b2b_micro_establishments",
            "share_b2b_small_establishments",
            "share_b2b_medium_establishments",
            "share_b2b_large_establishments",
            "share_b2b_medium_large_establishments"
        ]
    ]
    .describe()
)

print("\nDESCRIPTIVOS SOLO ENTRE MUNICIPIOS CON B2B")
print("=" * 100)
display(descriptive_with_b2b)


# ------------------------------------------------------------
# 8. Guardar outputs
# ------------------------------------------------------------

validation_summary_path = tables_path / "denue_b2b_medium_large_validation_summary.csv"
top_b2b_medium_large_path = tables_path / "top_municipalities_b2b_medium_large.csv"
b2b_only_micro_small_path = tables_path / "top_municipalities_b2b_only_micro_small.csv"
descriptive_with_b2b_path = tables_path / "denue_b2b_size_descriptive_only_municipalities_with_b2b.csv"

validation_summary.to_csv(
    validation_summary_path,
    index=False,
    encoding="utf-8-sig"
)

top_b2b_medium_large.to_csv(
    top_b2b_medium_large_path,
    index=False,
    encoding="utf-8-sig"
)

b2b_only_micro_small.to_csv(
    b2b_only_micro_small_path,
    index=False,
    encoding="utf-8-sig"
)

descriptive_with_b2b.to_csv(
    descriptive_with_b2b_path,
    encoding="utf-8-sig"
)

print("\nARCHIVOS GUARDADOS")
print("=" * 100)
print(validation_summary_path)
print(top_b2b_medium_large_path)
print(b2b_only_micro_small_path)
print(descriptive_with_b2b_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base cargada
Filas: 4,905
Columnas: 34


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_total_establishments,denue_total_scian_classes,denue_manufacturing_establishments,denue_logistics_storage_establishments,...,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,share_b2b_micro_establishments,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments
0,2018,01,AGUASCALIENTES,001,Aguascalientes,01001,6740,281,3802,352,...,2262,536,99,41,140,0.769912,0.182437,0.033696,0.013955,0.047651
1,2018,01,AGUASCALIENTES,002,Asientos,01002,72,20,53,2,...,18,1,0,0,0,0.947368,0.052632,0.000000,0.000000,0.000000
2,2018,01,AGUASCALIENTES,003,Calvillo,01003,291,61,205,7,...,79,5,2,0,2,0.918605,0.058140,0.023256,0.000000,0.023256
3,2018,01,AGUASCALIENTES,004,Cosío,01004,44,16,31,1,...,12,1,0,0,0,0.923077,0.076923,0.000000,0.000000,0.000000
4,2018,01,AGUASCALIENTES,005,Jesús María,01005,854,160,651,58,...,146,40,14,3,17,0.719212,0.197044,0.068966,0.014778,0.083744



RESUMEN DE VALIDACIÓN POR AÑO


,year,n_municipalities,municipalities_without_b2b,municipalities_with_b2b,municipalities_with_b2b_only_micro_small,municipalities_with_b2b_medium_large,municipalities_with_b2b_large,total_b2b_establishments,total_b2b_micro,total_b2b_small,...,share_municipalities_without_b2b,share_municipalities_with_b2b,share_municipalities_with_b2b_only_micro_small,share_municipalities_with_b2b_medium_large,share_municipalities_with_b2b_large,share_b2b_micro_national,share_b2b_small_national,share_b2b_medium_national,share_b2b_large_national,share_b2b_medium_large_national
0,2018,2442,170,2272,1613,659,382,248336,196793,37942,...,0.069615,0.930385,0.660524,0.269861,0.156429,0.792447,0.152785,0.034800,0.019969,0.054769
1,2023,2463,208,2255,1524,731,392,228907,168580,44657,...,0.084450,0.915550,0.618758,0.296793,0.159156,0.736456,0.195088,0.042996,0.025460,0.068456



TOP 30 MUNICIPIOS POR B2B MEDIUM/LARGE, POR AÑO


...


TOP MUNICIPIOS CON B2B PERO SOLO MICRO/SMALL


,year,entidad_name,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,share_b2b_micro_establishments,share_b2b_small_establishments
1034,2018,OAXACA,Heroica Ciudad de Huajuapan de León,20039,375,351,24,0,0,0.936000,0.064000
775,2018,MÉXICO,Zumpango,15120,319,294,25,0,0,0.921630,0.078370
272,2018,CIUDAD DE MÉXICO,Milpa Alta,09009,236,234,2,0,0,0.991525,0.008475
538,2018,JALISCO,Arandas,14008,187,176,11,0,0,0.941176,0.058824
2018,2018,TLAXCALA,Huamantla,29013,174,163,11,0,0,0.936782,0.063218
743,2018,MÉXICO,Tenancingo,15088,169,160,9,0,0,0.946746,0.053254
756,2018,MÉXICO,Tianguistenco,15101,167,159,8,0,0,0.952096,0.047904
545,2018,JALISCO,Autlán de Navarro,14015,159,141,18,0,0,0.886792,0.113208
2225,2018,VERACRUZ DE IGNACIO DE LA LLAVE,Álamo Temapache,30160,142,138,4,0,0,0.971831,0.028169
94,2018,CHIAPAS,Cintalapa,07017,136,120,16,0,0,0.882353,0.117647



DESCRIPTIVOS SOLO ENTRE MUNICIPIOS CON B2B


denue_b2b_support_establishments                                          \
                                count        mean         std  min  25%   50%   
year                                                                            
2018                           2272.0  109.302817  418.268448  1.0  4.0  13.0   
2023                           2255.0  101.510865  377.937922  1.0  4.0  13.0   

                   denue_b2b_medium_large_establishments            ...  \
       75%     max                                 count      mean  ...   
year                                                                ...   
2018  46.0  7008.0                                2272.0  5.986356  ...   
2023  45.0  5649.0                                2255.0  6.949002  ...   

     share_b2b_large_establishments       \
                                75%  max   
year                                       
2018                            0.0  1.0   
2023                            0.0  1.0   

     share_b2b_medium_large_establishments                                     \
                                     count      mean       std  min  25%  50%   
year                                                                            
2018                                2272.0  0.016864  0.046502  0.0  0.0  0.0   
2023                                2255.0  0.023302  0.063784  0.0  0.0  0.0   

                     
           75%  max  
year                 
2018  0.012862  1.0  
2023  0.022832  1.0  

[2 rows x 64 columns]


ARCHIVOS GUARDADOS
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_b2b_medium_large_validation_summary.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/top_municipalities_b2b_medium_large.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/top_municipalities_b2b_only_micro_small.csv
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/denue_b2b_size_descriptive_only_municipalities_with_b2b.csv


In [4]:
# ============================================================
# VALIDACIÓN DEL MÓDULO MUNICIPAL DENUE
# Proyecto: Nearshoring_Project
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

input_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_denue_support_summary.csv"
)

tables_path = Path(
    "/content/drive/MyDrive/Nearshoring_Project/outputs/tables"
)

tables_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Cargar base
# ------------------------------------------------------------

denue_mun = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

denue_mun["year"] = pd.to_numeric(denue_mun["year"], errors="coerce").astype("Int64")
denue_mun["entidad_id"] = denue_mun["entidad_id"].astype(str).str.zfill(2)
denue_mun["municipio_id"] = denue_mun["municipio_id"].astype(str).str.zfill(3)
denue_mun["geo_key"] = denue_mun["entidad_id"] + denue_mun["municipio_id"]

print("Base cargada")
print("=" * 100)
print(f"Filas: {denue_mun.shape[0]:,}")
print(f"Columnas: {denue_mun.shape[1]:,}")
display(denue_mun.head())


# ------------------------------------------------------------
# 3. Validación de identidades aritméticas
# ------------------------------------------------------------

denue_mun["b2b_size_sum"] = (
    denue_mun["denue_b2b_micro_establishments"] +
    denue_mun["denue_b2b_small_establishments"] +
    denue_mun["denue_b2b_medium_establishments"] +
    denue_mun["denue_b2b_large_establishments"]
)

denue_mun["b2b_size_diff"] = (
    denue_mun["denue_b2b_support_establishments"] -
    denue_mun["b2b_size_sum"]
)

denue_mun["medium_large_check"] = (
    denue_mun["denue_b2b_medium_establishments"] +
    denue_mun["denue_b2b_large_establishments"]
)

denue_mun["medium_large_diff"] = (
    denue_mun["denue_b2b_medium_large_establishments"] -
    denue_mun["medium_large_check"]
)

arithmetic_validation = (
    denue_mun
    .groupby("year")
    .agg(
        n_municipalities=("geo_key", "nunique"),
        total_b2b=("denue_b2b_support_establishments", "sum"),
        b2b_size_sum=("b2b_size_sum", "sum"),
        total_b2b_size_diff=("b2b_size_diff", "sum"),
        total_medium_large=("denue_b2b_medium_large_establishments", "sum"),
        medium_large_check=("medium_large_check", "sum"),
        total_medium_large_diff=("medium_large_diff", "sum")
    )
    .reset_index()
)

print("\nVALIDACIÓN ARITMÉTICA")
print("=" * 100)
display(arithmetic_validation)


# ------------------------------------------------------------
# 4. Banderas interpretativas
# ------------------------------------------------------------

denue_mun["has_b2b_support"] = (
    denue_mun["denue_b2b_support_establishments"] > 0
).astype(int)

denue_mun["has_no_b2b_support"] = (
    denue_mun["denue_b2b_support_establishments"] == 0
).astype(int)

denue_mun["has_b2b_only_micro_small"] = (
    (denue_mun["denue_b2b_support_establishments"] > 0) &
    (denue_mun["denue_b2b_medium_large_establishments"] == 0)
).astype(int)

denue_mun["has_b2b_medium_large"] = (
    denue_mun["denue_b2b_medium_large_establishments"] > 0
).astype(int)

denue_mun["has_b2b_large"] = (
    denue_mun["denue_b2b_large_establishments"] > 0
).astype(int)


# ------------------------------------------------------------
# 5. Resumen de cobertura territorial
# ------------------------------------------------------------

territorial_validation = (
    denue_mun
    .groupby("year")
    .agg(
        n_municipalities=("geo_key", "nunique"),
        municipalities_without_b2b=("has_no_b2b_support", "sum"),
        municipalities_with_b2b=("has_b2b_support", "sum"),
        municipalities_with_b2b_only_micro_small=("has_b2b_only_micro_small", "sum"),
        municipalities_with_b2b_medium_large=("has_b2b_medium_large", "sum"),
        municipalities_with_b2b_large=("has_b2b_large", "sum")
    )
    .reset_index()
)

territorial_validation["share_municipalities_without_b2b"] = (
    territorial_validation["municipalities_without_b2b"] /
    territorial_validation["n_municipalities"]
)

territorial_validation["share_municipalities_with_b2b"] = (
    territorial_validation["municipalities_with_b2b"] /
    territorial_validation["n_municipalities"]
)

territorial_validation["share_municipalities_with_b2b_only_micro_small"] = (
    territorial_validation["municipalities_with_b2b_only_micro_small"] /
    territorial_validation["n_municipalities"]
)

territorial_validation["share_municipalities_with_b2b_medium_large"] = (
    territorial_validation["municipalities_with_b2b_medium_large"] /
    territorial_validation["n_municipalities"]
)

territorial_validation["share_municipalities_with_b2b_large"] = (
    territorial_validation["municipalities_with_b2b_large"] /
    territorial_validation["n_municipalities"]
)

territorial_validation["share_b2b_medium_large_among_municipalities_with_b2b"] = (
    territorial_validation["municipalities_with_b2b_medium_large"] /
    territorial_validation["municipalities_with_b2b"]
)

print("\nVALIDACIÓN TERRITORIAL")
print("=" * 100)
display(territorial_validation)


# ------------------------------------------------------------
# 6. Composición nacional de establecimientos B2B por tamaño
# ------------------------------------------------------------

national_size_composition = (
    denue_mun
    .groupby("year")
    .agg(
        total_b2b=("denue_b2b_support_establishments", "sum"),
        b2b_micro=("denue_b2b_micro_establishments", "sum"),
        b2b_small=("denue_b2b_small_establishments", "sum"),
        b2b_medium=("denue_b2b_medium_establishments", "sum"),
        b2b_large=("denue_b2b_large_establishments", "sum"),
        b2b_medium_large=("denue_b2b_medium_large_establishments", "sum")
    )
    .reset_index()
)

for col in ["micro", "small", "medium", "large", "medium_large"]:
    national_size_composition[f"share_b2b_{col}"] = (
        national_size_composition[f"b2b_{col}"] /
        national_size_composition["total_b2b"]
    )

print("\nCOMPOSICIÓN NACIONAL DE B2B POR TAMAÑO")
print("=" * 100)
display(national_size_composition)


# ------------------------------------------------------------
# 7. Top municipios por B2B medium/large
# ------------------------------------------------------------

top_medium_large = (
    denue_mun
    .sort_values(
        ["year", "denue_b2b_medium_large_establishments"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    [
        [
            "year",
            "entidad_name",
            "municipio_name",
            "geo_key",
            "denue_b2b_support_establishments",
            "denue_b2b_medium_large_establishments",
            "denue_b2b_large_establishments",
            "share_b2b_medium_large_establishments",
            "share_b2b_large_establishments",
            "denue_logistics_storage_establishments",
            "denue_professional_technical_establishments",
            "denue_business_support_establishments"
        ]
    ]
)

print("\nTOP 30 MUNICIPIOS POR B2B MEDIUM/LARGE")
print("=" * 100)
display(top_medium_large)


# ------------------------------------------------------------
# 8. Municipios con B2B, pero solo micro/small
# ------------------------------------------------------------

top_only_micro_small = (
    denue_mun
    .query("denue_b2b_support_establishments > 0 and denue_b2b_medium_large_establishments == 0")
    .sort_values(
        ["year", "denue_b2b_support_establishments"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    [
        [
            "year",
            "entidad_name",
            "municipio_name",
            "geo_key",
            "denue_b2b_support_establishments",
            "denue_b2b_micro_establishments",
            "denue_b2b_small_establishments",
            "share_b2b_micro_establishments",
            "share_b2b_small_establishments"
        ]
    ]
)

print("\nTOP MUNICIPIOS CON B2B PERO SOLO MICRO/SMALL")
print("=" * 100)
display(top_only_micro_small)


# ------------------------------------------------------------
# 9. Guardar tablas de validación
# ------------------------------------------------------------

arithmetic_validation.to_csv(
    tables_path / "denue_validation_arithmetic_checks.csv",
    index=False,
    encoding="utf-8-sig"
)

territorial_validation.to_csv(
    tables_path / "denue_validation_territorial_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

national_size_composition.to_csv(
    tables_path / "denue_validation_national_b2b_size_composition.csv",
    index=False,
    encoding="utf-8-sig"
)

top_medium_large.to_csv(
    tables_path / "denue_validation_top_b2b_medium_large_municipalities.csv",
    index=False,
    encoding="utf-8-sig"
)

top_only_micro_small.to_csv(
    tables_path / "denue_validation_top_b2b_only_micro_small_municipalities.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nARCHIVOS DE VALIDACIÓN GUARDADOS EN:")
print(tables_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base cargada
Filas: 4,905
Columnas: 34


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_total_establishments,denue_total_scian_classes,denue_manufacturing_establishments,denue_logistics_storage_establishments,...,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,share_b2b_micro_establishments,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments
0,2018,01,AGUASCALIENTES,001,Aguascalientes,01001,6740,281,3802,352,...,2262,536,99,41,140,0.769912,0.182437,0.033696,0.013955,0.047651
1,2018,01,AGUASCALIENTES,002,Asientos,01002,72,20,53,2,...,18,1,0,0,0,0.947368,0.052632,0.000000,0.000000,0.000000
2,2018,01,AGUASCALIENTES,003,Calvillo,01003,291,61,205,7,...,79,5,2,0,2,0.918605,0.058140,0.023256,0.000000,0.023256
3,2018,01,AGUASCALIENTES,004,Cosío,01004,44,16,31,1,...,12,1,0,0,0,0.923077,0.076923,0.000000,0.000000,0.000000
4,2018,01,AGUASCALIENTES,005,Jesús María,01005,854,160,651,58,...,146,40,14,3,17,0.719212,0.197044,0.068966,0.014778,0.083744



VALIDACIÓN ARITMÉTICA


,year,n_municipalities,total_b2b,b2b_size_sum,total_b2b_size_diff,total_medium_large,medium_large_check,total_medium_large_diff
0,2018,2442,248336,248336,0,13601,13601,0
1,2023,2463,228907,228907,0,15670,15670,0



VALIDACIÓN TERRITORIAL


...


TOP 30 MUNICIPIOS POR B2B MEDIUM/LARGE


,year,entidad_name,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,denue_b2b_large_establishments,share_b2b_medium_large_establishments,share_b2b_large_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments
279,2018,CIUDAD DE MÉXICO,Miguel Hidalgo,09016,3945,849,310,0.215209,0.078580,353,2510,1082
277,2018,CIUDAD DE MÉXICO,Benito Juárez,09014,4296,582,178,0.135475,0.041434,299,2983,1014
278,2018,CIUDAD DE MÉXICO,Cuauhtémoc,09015,6122,580,218,0.094740,0.035609,459,4022,1641
984,2018,NUEVO LEÓN,Monterrey,19039,4737,537,218,0.113363,0.046021,627,2543,1567
569,2018,JALISCO,Guadalajara,14039,7008,455,161,0.064926,0.022974,817,3751,2440
273,2018,CIUDAD DE MÉXICO,Álvaro Obregón,09010,1679,276,109,0.164384,0.064920,161,958,560
1788,2018,QUERÉTARO,Querétaro,22014,3543,259,102,0.073102,0.028789,427,1809,1307
1671,2018,PUEBLA,Puebla,21114,5014,249,86,0.049661,0.017152,611,2303,2100
14,2018,BAJA CALIFORNIA,Tijuana,02004,4275,238,80,0.055673,0.018713,782,1905,1588
339,2018,GUANAJUATO,León,11020,4220,232,100,0.054976,0.023697,539,1837,1844



TOP MUNICIPIOS CON B2B PERO SOLO MICRO/SMALL


,year,entidad_name,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_micro_establishments,denue_b2b_small_establishments,share_b2b_micro_establishments,share_b2b_small_establishments
1034,2018,OAXACA,Heroica Ciudad de Huajuapan de León,20039,375,351,24,0.936000,0.064000
775,2018,MÉXICO,Zumpango,15120,319,294,25,0.921630,0.078370
272,2018,CIUDAD DE MÉXICO,Milpa Alta,09009,236,234,2,0.991525,0.008475
538,2018,JALISCO,Arandas,14008,187,176,11,0.941176,0.058824
2018,2018,TLAXCALA,Huamantla,29013,174,163,11,0.936782,0.063218
743,2018,MÉXICO,Tenancingo,15088,169,160,9,0.946746,0.053254
756,2018,MÉXICO,Tianguistenco,15101,167,159,8,0.952096,0.047904
545,2018,JALISCO,Autlán de Navarro,14015,159,141,18,0.886792,0.113208
2225,2018,VERACRUZ DE IGNACIO DE LA LLAVE,Álamo Temapache,30160,142,138,4,0.971831,0.028169
94,2018,CHIAPAS,Cintalapa,07017,136,120,16,0.882353,0.117647



ARCHIVOS DE VALIDACIÓN GUARDADOS EN:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables
